# Publish the gearbox assembly

Kernel: **SysML** -- check the top right, not Python.

This is the two-part reference assembly: Housing and Cover under one
assembly, a shared bore as the interface dimension, and an assembly-level
mass budget. Its CAD counterparts already exist as `cad/gearbox.py` and
`fixtures/step/gearbox.step`.

Why it matters: with one part and two attributes, a mapping proposer that
gets both right has shown nothing. This model is what makes
`make propose-score` a measurement rather than a mechanism.

The model below is `models/gearbox.sysml` verbatim -- keep the two in step.


In [1]:
package Gearbox {
    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;

    part def Housing {
        attribute length     : LengthValue;
        attribute width      : LengthValue;
        attribute height     : LengthValue;
        attribute boreRadius : LengthValue;
        attribute mass       : MassValue;
    }

    part def Cover {
        attribute length     : LengthValue;
        attribute width      : LengthValue;
        attribute thickness  : LengthValue;
        attribute boreRadius : LengthValue;
        attribute mass       : MassValue;
    }

    part def GearboxAssembly {
        attribute mass : MassValue;
    }

    requirement def MaxHousingLength {
        doc /* The gearbox housing must not exceed 95 mm in length. */
        subject housing : Housing;
        attribute maxLength : LengthValue = 95 [mm];
        require constraint { housing.length <= maxLength }
    }

    requirement def MaxCoverThickness {
        doc /* The cover must not exceed 8 mm in thickness. */
        subject cover : Cover;
        attribute maxThickness : LengthValue = 8 [mm];
        require constraint { cover.thickness <= maxThickness }
    }

    requirement def MinBoreRadius {
        doc /* The bore must be at least 8 mm in radius -- the shaft interface.
               Housing and cover have to agree on this; today each is checked
               against the agreed figure separately. */
        subject housing : Housing;
        attribute minRadius : LengthValue = 8 [mm];
        require constraint { housing.boreRadius >= minRadius }
    }

    requirement def MaxAssemblyMass {
        doc /* The assembled gearbox must not exceed 1.5 kg. This is the budget
               the components are allocated out of. */
        subject assembly : GearboxAssembly;
        attribute maxMass : MassValue = 1.5 [kg];
        require constraint { assembly.mass <= maxMass }
    }

    part gearbox : GearboxAssembly {
        part housing : Housing;
        part cover   : Cover;
    }

    requirement housingLengthCheck  : MaxHousingLength;
    requirement coverThicknessCheck : MaxCoverThickness;
    requirement boreRadiusCheck     : MinBoreRadius;
    requirement assemblyMassCheck   : MaxAssemblyMass;

    // The links the extractor resolves to learn which part each requirement is
    // about. Without these, every requirement's subject falls back to the
    // definition's own subject parameter -- see extract.resolve_satisfies.
    satisfy housingLengthCheck  by gearbox.housing;
    satisfy coverThicknessCheck by gearbox.cover;
    satisfy boreRadiusCheck     by gearbox.housing;
    satisfy assemblyMassCheck   by gearbox;
}


Package Gearbox (67ccd166-e483-4520-be0a-1ae6e9f436eb)


In [2]:
%publish Gearbox


API base path: http://sysml-api:9000
Processing...
Posting commit. Additions: 370, Deletions: 0, Modified elements: 0
Successfully posted commit: 9fca5f95-cf03-4ae4-887a-b3724d72e522 on branch: (main) f40924a4-d778-47c6-b467-b12e7895f89b



Saved to Project Gearbox (99368cde-af15-4119-a3d1-2c7fb27c8b21)


If `%publish` returns a project id, the model is on the server. The first
output line must read `http://sysml-api:9000` -- if it says
`sysml2.intercax.com`, it went to the public community server instead.

Then, from the devcontainer:

```bash
python -m sysmlcad.catalog api      # three parts, ten attributes
python -m sysmlcad.verify api       # the gearbox requirements too
```
